In [1]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy import optimize

from skimage import io as skio
from skimage.color import rgb2gray, rgb2hed
from skimage.filters import gaussian, median, sobel, frangi, threshold_otsu
from skimage.feature import structure_tensor
from skimage.measure import label, regionprops_table
from skimage.morphology import disk, remove_small_objects
from skimage.restoration import denoise_nl_means, estimate_sigma
from skimage.transform import resize
from sklearn.decomposition import NMF
from sklearn.neighbors import NearestNeighbors
from pyimzml.ImzMLParser import ImzMLParser

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
from pathlib import Path
from scipy.interpolate import RBFInterpolator
from scipy.optimize import minimize
from skimage import transform, filters
try:
    import SimpleITK as sitk
    SITK_AVAILABLE = True
except ImportError:
    SITK_AVAILABLE = False
    print("WARNING: SimpleITK not installed. B-spline registration unavailable.")
    print("Install with: pip install SimpleITK")
from pyimzml.ImzMLParser import ImzMLParser
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import pandas as pd
import warnings
warnings.filterwarnings('ignore')


In [2]:
# step 1: Run Intensity based registration using Taurine as our reference ion. This will give us a rough global alignment between the 
# MALDI and H&E images, which we can then refine with our structural maps.

import sys
from pathlib import Path

# Adjust this to point to your project root
project_root = Path("/home/kia/MALDI_Metabolomics")
sys.path.append(str(project_root))

from scripts.final_coarse_reg import run_registration_pipeline

FULL_HE_PATH    = "high_res_MSI/D2_10x_originalExport.tif"
MALDI_IMG_PATH = "img_folder/Taurine_img_withoutborders.tif"
GLOBALREG_OUTPUT_DIR = "registration_outputs"

USE_SAVED_LANDMARKS = True
LANDMARKS = {
    'he': [
        [7612.3, 4264.6], [8810.7, 6485.2], [10424.6, 7485.0],
        [10578.1, 6496.4], [10720.4, 5054.7], [9851.7, 5159.6],
        [9402.3, 4523.0], [8424.9, 4069.9],
    ],
    'maldi': [
        [97.2,  99.8],  [275.8, 442.5], [515.8, 599.4],
        [544.6, 445.1], [566.1, 227.3], [437.3, 243.0],
        [368.6, 145.6], [220.8,  72.3],
    ]
}

registration = run_registration_pipeline(
        he_path=FULL_HE_PATH,
        maldi_path=MALDI_IMG_PATH,
        n_landmarks=8,
        save_coords=True,
        grid_spacing=1,
        tissue_only=True,
        output_dir=GLOBALREG_OUTPUT_DIR,
        use_napari=True,
        use_full_affine=False,
        use_saved_landmarks=USE_SAVED_LANDMARKS,
        landmarks=LANDMARKS,
        max_residual_px=None,
        auto_boundary_align=True,
    )


MALDI-MSI TO H&E REGISTRATION PIPELINE

Output directory: /home/kia/MALDI_Metabolomics/registration_outputs

Step 1/5: Loading images...
  Loaded via tifffile: shape=(12468, 15297, 3)
Loaded H&E image:   (12468, 15297)
Loaded MALDI image: (712, 786)

Step 2a/5: Auto tissue boundary alignment...

Auto-aligning tissue boundaries (exhaustive rotation search)...
  H&E tissue mask: 34.2% tissue (saturation-based)
  MALDI tissue mask: 47.9% tissue (variance+Otsu)
  Saved tissue masks -> registration_outputs (check if they look right)
  H&E centroid:   (8127, 6370)
  MALDI centroid: (400, 327)
  Scale estimate: 15.595

  Exhaustive search: 180 angles at 1/32 (478x389 px)...
  Top 5 candidates:
       6.0 deg  IoU=0.7059
       8.0 deg  IoU=0.7051
       4.0 deg  IoU=0.7049
       2.0 deg  IoU=0.7024
      10.0 deg  IoU=0.7014
  Best: 6.0 deg  IoU=0.7059

  Powell refinement at 1/8 (1912x1558 px)...
  Refined: scale=16.0353, rot=6.43 deg, IoU=0.7255, evals=679

Step 2b/5: Landmark selection..

In [3]:
import numpy as np
import pandas as pd
import skimage.io as skio
import SimpleITK as sitk
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Load ROI coordinates from your mapping file
# ---------------------------------------------------------
roi_he = pd.read_csv('registration_outputs/coordinate_mapping.csv')

x_min = int(roi_he['he_x'].min())
x_max = int(roi_he['he_x'].max())
y_min = int(roi_he['he_y'].min())
y_max = int(roi_he['he_y'].max())

# ---------------------------------------------------------
# Load full‑resolution H&E image
# ---------------------------------------------------------
he_img = skio.imread(FULL_HE_PATH)

# ---------------------------------------------------------
# Crop using the bounding box
# ---------------------------------------------------------
he_cropped_img = he_img[y_min:y_max, x_min:x_max]

# ---------------------------------------------------------
# Store the global offset (this preserves the coordinate system)
# ---------------------------------------------------------

# he_cropped is your cropped H&E image
H, W = he_cropped_img.shape[:2]

# Local coordinate grid
xs = np.arange(W)
ys = np.arange(H)
X_local, Y_local = np.meshgrid(xs, ys)

# Global offset from original H&E
he_offset = np.array([x_min, y_min])

# Global coordinate grid
X_global = X_local + he_offset[0]
Y_global = Y_local + he_offset[1]


# check the range of the global coordinates
print("Global X range:", X_global.min(), "to", X_global.max())
print("Global Y range:", Y_global.min(), "to", Y_global.max())
he_cropped_coords = np.stack([X_global, Y_global], axis=-1)
print(he_cropped_coords.shape)

# now time to create a flattened version of the global coordinates for use in registration
maldi_bb_coords = he_cropped_coords.reshape(-1, 2)
print(maldi_bb_coords[:3, :])
print(maldi_bb_coords.shape)

# save the cropped image 
skio.imsave(fname='/home/kia/MALDI_Metabolomics/registration_outputs/he_cropped.tif', arr=he_cropped_img)

Global X range: 6979 to 12183
Global Y range: 3606 to 8201
(4596, 5205, 2)
[[6979 3606]
 [6980 3606]
 [6981 3606]]
(23922180, 2)


In [4]:
def compute_correspondence_table(maldi: dict,
                                  he: dict,
                                  transform: sitk.Transform,
                                  cropped_coords: np.array,
                                  imzml_path: str) -> tuple:
    """
    Build a MALDI→H&E pixel correspondence table — one row per spectrum.

    Returns (table, columns) where:
      table   : [N, 4] float64 array
      columns : ["maldi_x", "maldi_y", "he_x", "he_y"]

    Columns:
      0  maldi_x        column in the MALDI native grid (0..W_m - 1)
      1  maldi_y        row in the MALDI native grid (0..H_m - 1)
      2  he_x           x in full-resolution H&E pixel space (fractional)
      3  he_y           y in full-resolution H&E pixel space (fractional)

    The MALDI→H&E direction uses the INVERSE of the registration transform
    (since the registration mapped fixed=H&E → moving=MALDI). For pure
    affine transforms this is computed analytically and vectorised; for
    composite transforms (B-spline) it falls back to per-point evaluation.
    """
    
    parser = ImzMLParser(imzml_path)
    coords = list(parser.coordinates)
    n = len(coords)

    x0, y0 = maldi["grid_origin"]
    dy_he, dx_he = he["downsample_factor"]

    imzml_x = np.array([c[0] for c in coords], dtype=np.float64)
    imzml_y = np.array([c[1] for c in coords], dtype=np.float64)
    grid_x  = imzml_x - x0
    grid_y  = imzml_y - y0

    # MALDI → H&E registration-grid coords via inverse transform.
    # Linear 2D transforms expose matrix/center/translation, so use the
    # analytic inverse and keep the table export fast.
    if all(hasattr(transform, name) for name in
           ("GetMatrix", "GetTranslation", "GetCenter")):
        M = np.array(transform.GetMatrix()).reshape(2, 2)
        t = np.array(transform.GetTranslation())
        c = np.array(transform.GetCenter())
        Minv = np.linalg.inv(M)
        pts = np.stack([grid_x, grid_y], axis=1)
        he_reg = (pts - t - c) @ Minv.T + c
    else: 
        raise TypeError ("No SITK object found for transform")

    # Registration-grid → full H&E pixel space (multiply by downsample factor)
    he_full_x = he_reg[:, 0] * dx_he + min(cropped_coords[:,0]) # offset
    he_full_y = he_reg[:, 1] * dy_he + min(cropped_coords[:,1])

    table = np.column_stack([grid_x, grid_y, he_full_x, he_full_y])
    columns = ["maldi_x", "maldi_y", "he_x", "he_y"]
    return table, columns

In [5]:
# time to see if I can nicely just insert the cropped coordinates into the refinement pipeline
import scripts.preprocessed_structuralmap as smap
# -----------------------------
# Global configuration variables
# -----------------------------
IMZML_PATH = "/home/kia/MALDI_Metabolomics/MSI_data_grant/Mass_Spec_data/20251012_old_liver.imzML"
HE_PATH    = "/home/kia/MALDI_Metabolomics/registration_outputs/he_cropped.tif"
OUTPUT_DIR = "/home/kia/MALDI_Metabolomics/mutual_inf_reg"

N_COMPONENTS = 8
MZ_BIN = 0.042
MZ_BIN_MIN_COUNT = 100

MALDI_UM_PER_PIXEL = 5.0
HE_UM_PER_PIXEL = None   # or a float

USE_HEMATOXYLIN = True
DENOISE_METHOD = "gaussian"   # gaussian | median | nl_means | none
DENOISE_SIGMA = 1.0
MEDIAN_RADIUS = 2

STRUCTURAL_COMBINE = "max"
REGISTRATION_FIELD = "edge"   # edge | anisotropy | vesselness
TRANSFORM_MODEL = "euler"     # ignored
USE_BSPLINE = False           # ignored

def run_pipeline(cropped_coords: np.array):
    out = Path(OUTPUT_DIR)
    out.mkdir(parents=True, exist_ok=True)

    # -------------------------
    # 1+2. MALDI data
    # -------------------------
    maldi = smap.build_maldi_data(
        IMZML_PATH,
        n_components=N_COMPONENTS,
        mz_bin_width=MZ_BIN,
        mz_bin_min_count=MZ_BIN_MIN_COUNT,
        pixel_size_um=MALDI_UM_PER_PIXEL,
        denoise_method=DENOISE_METHOD,
        gaussian_sigma=DENOISE_SIGMA,
        median_radius=MEDIAN_RADIUS,
        structural_combine=STRUCTURAL_COMBINE,
        registration_field=REGISTRATION_FIELD,
    )

    smap.save_component_denoising_qc(
        maldi["components"],
        maldi["denoised_components"],
        out / "nmf_component_denoising.png",
    )

    smap.save_structural_fields(
        maldi["structural_fields"],
        out / "maldi_structural_fields.png",
        "MALDI structural fields from NMF components",
    )

    # -------------------------
    # 3. H&E
    # -------------------------
    he = smap.prepare_he(
        HE_PATH,
        maldi["grid_shape"],
        use_hematoxylin=USE_HEMATOXYLIN,
        pixel_size_um=HE_UM_PER_PIXEL,
        maldi_um_per_pixel=MALDI_UM_PER_PIXEL,
        registration_field=REGISTRATION_FIELD,
    )

    smap.save_image(
        he["registration_grid"],
        out / "he_at_registration_grid.png",
        cmap="gray",
        title="H&E at MALDI grid",
    )

    smap.save_structural_fields(
        he["structural_fields"],
        out / "he_structural_fields.png",
        "H&E hematoxylin structural fields",
    )

    # -------------------------
    # Vessel detection
    # -------------------------
    maldi_vessels = smap.detect_vessels(maldi["structural_fields"]["vesselness"])
    he_vessels = smap.detect_vessels(he["structural_fields"]["vesselness"])

    maldi_vessels["table"].to_csv(out / "maldi_vessels.csv", index=False)
    he_vessels["table"].to_csv(out / "he_vessels.csv", index=False)

    smap.save_vessel_overlay(
        he["structural_fields"]["edge"],
        he_vessels,
        maldi_vessels,
        out / "vessel_overlay_pre.png",
        "Pre-registration vessel centers and contours",
    )

    smap.save_difference(
        maldi["registration_image"],
        he["registration_image"],
        out / "structural_difference_pre.png",
        title="Pre-registration structural-field difference",
    )

    # -------------------------
    # Pre-registration coordinate export
    # -------------------------
    print("[4/7] Exporting pre-registration coordinate map")

    identity_tx = sitk.Euler2DTransform()
    identity_tx.SetCenter((
        (maldi["grid_shape"][1] - 1) / 2.0,
        (maldi["grid_shape"][0] - 1) / 2.0,
    ))
    identity_tx.SetAngle(0.0)
    identity_tx.SetTranslation((0.0, 0.0))

    corr_table, corr_cols = smap.compute_correspondence_table(
        maldi, he, identity_tx, IMZML_PATH
    )

    smap.save_correspondence_csv(
        corr_table,
        corr_cols,
        out / "pre_registration_coordinate_map.csv",
    )

    # -------------------------
    # 4. Direct biological optimization
    # -------------------------
    fixed  = smap.np_to_sitk(he["registration_image"])
    moving = smap.np_to_sitk(maldi["registration_image"])

    he_tissue_mask = smap.build_tissue_mask(he["structural_fields"])
    maldi_tissue_mask = smap.build_tissue_mask(maldi["structural_fields"])

    smap.save_image(maldi_tissue_mask.astype(np.float32), out / "maldi_tissue_mask.png",
               cmap="gray", title="MALDI structural tissue mask")
    smap.save_image(he_tissue_mask.astype(np.float32), out / "he_tissue_mask.png",
               cmap="gray", title="H&E structural tissue mask")

    final_tx, objective_trace, objective_stats = smap.optimize_biological_transform(
        fixed_image=he["registration_image"],
        moving_image=maldi["registration_image"],
        fixed_mask=he_tissue_mask,
        moving_mask=maldi_tissue_mask,
        maldi_vessels=maldi_vessels["table"],
        he_vessels=he_vessels["table"],
        fixed_sitk=fixed,
        moving_sitk=moving,
    )

    objective_trace.to_csv(out / "biological_objective_trace.csv", index=False)
    pd.DataFrame([objective_stats]).to_csv(out / "biological_objective_summary.csv", index=False)

    # -------------------------
    # 6. Diagnostics
    # -------------------------
    warped_structural = smap.warp_to_fixed(moving, fixed, final_tx)

    smap.save_overlay(
        warped_structural,
        he["registration_image"],
        out / "registered_overlay.png",
    )

    smap.save_difference(
        warped_structural,
        he["registration_image"],
        out / "structural_difference_post.png",
        title="Post-registration structural-field difference",
    )

    displacement_df, displacement_stats = smap.vessel_displacement_stats(
        maldi_vessels["table"],
        he_vessels["table"],
        final_tx,
        he["registration_image"].shape,
    )

    displacement_df.to_csv(out / "vessel_displacements.csv", index=False)

    # -------------------------
    # 7. Final coordinate export
    # -------------------------
    corr_table, corr_cols = compute_correspondence_table(
        maldi, he, transform=final_tx, cropped_coords=cropped_coords, imzml_path=IMZML_PATH
    )

    smap.save_correspondence_csv(
        corr_table,
        corr_cols,
        out / "maldi_to_he_table.csv",
    )

    print("\n✓ All outputs in", out.resolve())

run_pipeline(maldi_bb_coords)

[1/7] Loaded 269511 spectra | grid 713×787 | 5.0 μm/pixel
[1/7] m/z [57.03, 648.33] | dense bin tolerance 0.042 Da | min_count=100
[1/7] Dense m/z binning kept 39,620,699 values in 515 bins
[1/7] Spectral matrix: (269511, 515)
[2/7] NMF with k=8
[2/7] NMF reconstruction error: 39177.137
[2/7] Denoising NMF components with method='gaussian'
[2/7] Computing MALDI structural fields from 8 NMF components
[3/7] H&E pixel size inferred at 0.766 μm (area-matched against MALDI)
[3/7] Resizing H&E (4596, 5205) → (713, 787) (downsample 6.4×6.6)
[3/7] Computing H&E structural fields
[4/7] Exporting pre-registration coordinate map
[7/7] Correspondence CSV → /home/kia/MALDI_Metabolomics/mutual_inf_reg/pre_registration_coordinate_map.csv (269511 rows)
[4/7] Identity biological objective = 4.3994
[4/7] Optimizing direct biological objective: vessel + structure + overlap
[4/7] Selected biological optimum from local: score=4.3936, vessel=31.619px, struct=0.367, overlap=0.427, angle=0.126, tx=0.018, ty=

In [6]:
# time to check the coordinate system output 
ccs = pd.read_csv('/home/kia/MALDI_Metabolomics/mutual_inf_reg/maldi_to_he_table.csv')
ccs

,maldi_x,maldi_y,he_x,he_y
0,162,0,8045.1426,3609.8070
1,163,0,8051.7563,3609.7929
2,162,1,8045.1571,3616.2530
3,163,1,8051.7708,3616.2389
4,164,1,8058.3845,3616.2247
...,...,...,...,...
269506,553,710,10641.4056,8180.9277
269507,554,710,10648.0193,8180.9135
269508,552,711,10634.8064,8187.3878
269509,553,711,10641.4201,8187.3737


In [ ]:
# Get values for the immune cells m/z=403.2628
def filter_mz_sf_df(imzml_path: str, targets: list, ppm: float = 15):
    """
    RAM‑efficient m/z filtering:
    - No explode()
    - No long DataFrame
    - Stream spectra one by one
    """
    parser = ImzMLParser(imzml_path)

    # Precompute tolerances
    targets = np.asarray(targets, dtype=float)
    tols = targets * ppm / 1e6

    # Accumulate per‑pixel sums
    coords = []
    sums = []

    for idx, coord in enumerate(parser.coordinates):
        mzs, intens = parser.getspectrum(idx)

        mzs = np.asarray(mzs, dtype=float)
        intens = np.asarray(intens, dtype=float)

        # Boolean mask for ANY target window
        mask = np.zeros_like(mzs, dtype=bool)
        for t, tol in zip(targets, tols):
            mask |= (mzs >= t - tol) & (mzs <= t + tol)

        # Sum intensities for this pixel
        sums.append(intens[mask].sum())
        coords.append(tuple(coord))

    # Build final DataFrame
    df = pd.DataFrame({
        "coordinates": coords,
        f"mz_{targets[0]:.4f}": sums
    })

    return df
filtered_data = filter_mz_sf_df(imzml_path=IMZML_PATH, targets=[403.2628])
filtered_data

,coordinates,mz_403.2628
0,"(163, 1, 1)",0.0
1,"(164, 1, 1)",0.0
2,"(163, 2, 1)",0.0
3,"(164, 2, 1)",0.0
4,"(165, 2, 1)",0.0
...,...,...
269506,"(554, 711, 1)",0.0
269507,"(555, 711, 1)",0.0
269508,"(553, 712, 1)",0.0
269509,"(554, 712, 1)",0.0


In [13]:
import scripts.MultimodalRegistration as mr 

mr.mz_to_napari_points(signal=filtered_data['mz_403.2628'].values, ccs=ccs, col_x='he_x', col_y='he_y', 
                       he_path='/home/kia/MALDI_Metabolomics/high_res_MSI/D2_10x_originalExport.tif',
                       point_size=6)

Rendering 269,511 points total, 12,042 with signal
